# Yoked RL Simulations

Simple interface for running RL simulations yoked to mouse behavior data.

**Usage:**
1. Configure simulation parameters in Section 2
2. Run all cells to execute simulations
3. Review validation results

## 1. Setup

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add paths for imports
sys.path.insert(0, '../code')  # For utils_latMaz
sys.path.insert(0, '.')  # For yoked_rl_runner

# Reload modules for development
import importlib

# Import utilities from code
from utils_latMaz import get_most_recent_file, get_now_str

# Import simulation runner (from this directory)
import yoked_rl_runner
importlib.reload(yoked_rl_runner)
from yoked_rl_runner import YokedRLRunner, SimConfig, AlgoConfig, MemoryConfig, quick_validate

print(f"Setup complete. Working directory: {os.getcwd()}")
print(f"Available models: {list(YokedRLRunner.MODEL_CLASSES.keys())}")

## 2. Configuration

Modify these parameters to customize your simulation runs.

In [ ]:
# =============================================
# SIMULATION CONFIGURATION
# =============================================

# Models to test
# SB3 agents: A2C, DQN, PPO, TRPO, QRDQN, RecurrentPPO
# Custom DRQN agents (Hausknecht & Stone 2015): DRQN_seq, DRQN_rand
# Planning agent (requires pomdp-py): POMCP
MODELS = ['A2C', 'DQN', 'PPO', 'TRPO', 'QRDQN', 'RecurrentPPO',
          'DRQN_seq', 'DRQN_rand']
# Uncomment to include POMCP (slow, ~500x slower than RL agents):
# MODELS.append('POMCP')

# History lengths for observation history concatenation experiments
# Note: RecurrentPPO, DRQN, and POMCP skip h_len > 0 (they have their own memory)
HISTORY_LENGTHS = [0, 10, 20]

# Observation/action configurations
OBS_CONFIGS = [
    {
        'name': 'ego',
        'obs_type': 'ego',
        'action_type': 'ego',
        'include_prev_action': True,
        'include_prev_reward': True,
    },
    {
        'name': 'allo',
        'obs_type': 'allo_latent',
        'action_type': 'allo_latent',
        'include_prev_action': False,
        'include_prev_reward': False,
    },
]

# Number of repeats per configuration
TARGET_REPEATS = 10

# Random seed base
SEED_BASE = 42

# Session filter (None = all sessions)
SESSION_FILTER = None  # or e.g., ['251220-165800', '251119-171644']

# Memory failsafe configuration
MEMORY_CONFIG = MemoryConfig(
    enabled=True,
    limit_gb=8.0,           # Absolute memory cap
    threshold_fraction=0.80, # Or 80% of total RAM, whichever is lower
    snapshot_every_n_sims=10,
    save_every_n_sims=50,   # Intermediate save for crash resilience
    verbose=True,
)

print("Configuration set:")
print(f"  Models: {MODELS}")
print(f"  History lengths: {HISTORY_LENGTHS}")
print(f"  Obs configs: {[c['name'] for c in OBS_CONFIGS]}")
print(f"  Target repeats: {TARGET_REPEATS}")
print(f"  Memory limit: {MEMORY_CONFIG.limit_gb} GB")
if 'POMCP' not in MODELS:
    pomcp_status = "available" if 'POMCP' in YokedRLRunner.MODEL_CLASSES else "not installed (pip install pomdp-py)"
    print(f"  POMCP: {pomcp_status} (uncomment to enable)")

## 3. Load Data

In [ ]:
# Load yoking data (maps sessions to maze configs)
yoking_path = get_most_recent_file('../yoked_dfs/*animal_to_agent_yoking_info*.csv')
print(f"Loading yoking data from: {yoking_path}")
yoking_df = pd.read_csv(yoking_path, dtype={'animal_ID': str})

# Extract exp_moment from csv paths
import re
def extract_moment(path):
    match = re.search(r'(\d{6}-\d{6})', str(path))
    return match.group(1) if match else None
yoking_df['exp_moment'] = yoking_df['csv_data_path'].apply(extract_moment)

print(f"Loaded {len(yoking_df)} sessions")
print(f"Unique mazes: {yoking_df[['adj_file', 'st_pos_file']].drop_duplicates().shape[0]}")

In [ ]:
# Load rewarded states data (for proper reward initialization)
rwd_states_path = get_most_recent_file('../data_out/*rewarded_states*.csv')
if rwd_states_path:
    print(f"Loading rewarded states from: {rwd_states_path}")
    rwd_states_df = pd.read_csv(rwd_states_path)
    print(f"Loaded rewarded states for {len(rwd_states_df)} sessions")
else:
    print("WARNING: No rewarded states file found. Using fallback (all nodes rewarded).")
    rwd_states_df = None

## 4. Run Simulations

In [ ]:
# Create runner
runner = YokedRLRunner(
    yoking_df=yoking_df,
    rewarded_states_df=rwd_states_df,
    maze_dir='../data_in/mazes',
    output_dir='../data_out/rl_sims',
)

# Create config
config = SimConfig(
    models=MODELS,
    history_lengths=HISTORY_LENGTHS,
    observation_configs=OBS_CONFIGS,
    seed_base=SEED_BASE,
    verbose=True,
)

print("Runner initialized. Ready to run simulations.")

In [ ]:
# Run simulations with memory monitoring
results_df = runner.run(
    config=config,
    target_repeats=TARGET_REPEATS,
    session_filter=SESSION_FILTER,
    memory_config=MEMORY_CONFIG,
)

## 5. Validate Results

In [ ]:
# Quick validation
validation = quick_validate(results_df)

print("=" * 50)
print("VALIDATION RESULTS")
print("=" * 50)
for key, value in validation.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.2f}")
    else:
        print(f"  {key}: {value}")

In [ ]:
# Summary by model
print("\nResults by Model:")
print(results_df.groupby('model')['total_reward'].agg(['mean', 'std', 'min', 'max', 'count']))

In [ ]:
# Summary by config
print("\nResults by Config:")
print(results_df.groupby(['model', 'config_name', 'history_len'])['total_reward'].mean().unstack())

## 6. Quick Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(results_df['total_reward'], bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(results_df['total_reward'].mean(), color='red', linestyle='--', label=f"Mean: {results_df['total_reward'].mean():.1f}")
axes[0].set_xlabel('Total Reward')
axes[0].set_ylabel('Count')
axes[0].set_title('Reward Distribution')
axes[0].legend()

# By model
model_means = results_df.groupby('model')['total_reward'].mean()
model_stds = results_df.groupby('model')['total_reward'].std()
axes[1].bar(model_means.index, model_means.values, yerr=model_stds.values, capsize=5, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Mean Total Reward')
axes[1].set_title('Performance by Model')

plt.tight_layout()
plt.show()